In [1]:
import os
import gc
import re
import json
import math
import random
import warnings

import numpy as np
import polars as pl

from pathlib import Path
from tqdm.auto import tqdm
from rapidfuzz import fuzz

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score

import lightgbm as lgb

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Polars:", pl.__version__)
print("LightGBM:", lgb.__version__)

d:\business_entity_resolution\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Polars: 1.44.2
LightGBM: 4.7.0


In [4]:
DATA_ROOT = Path(".")

TRAIN_DIR = DATA_ROOT / "dataset" / "train"
TEST_DIR = DATA_ROOT / "dataset" / "test"

TRAIN_S1 = TRAIN_DIR / "train_source1.tsv"
TRAIN_S2 = TRAIN_DIR / "train_source2.tsv"
TRAIN_S3 = TRAIN_DIR / "train_source3.tsv"
GROUND_TRUTH = TRAIN_DIR / "train_ground_truth.tsv"

TEST_S1 = TEST_DIR / "test_source1.tsv"
TEST_S2 = TEST_DIR / "test_source2.tsv"
TEST_S3 = TEST_DIR / "test_source3.tsv"

OUTPUT_DIR = DATA_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Train directory:", TRAIN_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Train directory: D:\business_entity_resolution\dataset\train
Output directory: D:\business_entity_resolution\output


In [5]:
files = [
    TRAIN_S1,
    TRAIN_S2,
    TRAIN_S3,
    GROUND_TRUTH,
    TEST_S1,
    TEST_S2,
    TEST_S3,
]

for path in files:
    if path.exists():
        size_gb = path.stat().st_size / (1024 ** 3)
        print(f"{path.name:30s} {size_gb:.3f} GB")
    else:
        print(f"MISSING: {path}")

train_source1.tsv              0.196 GB
train_source2.tsv              0.456 GB
train_source3.tsv              0.469 GB
train_ground_truth.tsv         0.118 GB
test_source1.tsv               0.163 GB
test_source2.tsv               0.474 GB
test_source3.tsv               0.471 GB


In [6]:
for path in [TRAIN_S1, TRAIN_S2, TRAIN_S3]:
    print(f"\n{'=' * 60}")
    print(path.name)
    
    df = pl.read_csv(
        path,
        separator="\t",
        n_rows=5,
        infer_schema_length=1000,
    )
    
    print(df)
    print("\nSchema:")
    print(df.schema)


train_source1.tsv
shape: (5, 4)
┌──────────────┬─────────────────────────┬─────────────────────────────────┬─────────┐
│ entity_id    ┆ business_name           ┆ business_address                ┆ country │
│ ---          ┆ ---                     ┆ ---                             ┆ ---     │
│ str          ┆ str                     ┆ str                             ┆ str     │
╞══════════════╪═════════════════════════╪═════════════════════════════════╪═════════╡
│ S1-925783039 ┆ Orelee's Barbershop     ┆ 1795 Westchester Drive, High P… ┆ US      │
│ S1-773889195 ┆ Prime Money             ┆ 17560 Ellis Road, Tahlequah, O… ┆ US      │
│ S1-377745466 ┆ B+ Retail Inc           ┆ 1712 Montebello Avenue, Phoeni… ┆ US      │
│ S1-133037285 ┆ Christ Chapel           ┆ 2100 Cameron Drive, Unit APART… ┆ US      │
│ S1-755362802 ┆ Prabhav Business Center ┆ 797, Lake Town Block A, Kolkat… ┆ India   │
└──────────────┴─────────────────────────┴─────────────────────────────────┴─────────┘

Schema:
S

In [7]:
s1_lazy = pl.scan_csv(
    TRAIN_S1,
    separator="\t",
    infer_schema_length=10000,
)

s2_lazy = pl.scan_csv(
    TRAIN_S2,
    separator="\t",
    infer_schema_length=10000,
)

s3_lazy = pl.scan_csv(
    TRAIN_S3,
    separator="\t",
    infer_schema_length=10000,
)

print(s1_lazy.collect_schema())
print(s2_lazy.collect_schema())
print(s3_lazy.collect_schema())

Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})
Schema({'entity_id': String, 'business_name': String, 'business_address': String, 'country': String})


In [8]:
def get_stats(lazy_df, name):
    result = (
        lazy_df
        .select([
            pl.len().alias("rows"),
            pl.col("business_name").null_count().alias("null_names"),
            pl.col("business_address").null_count().alias("null_addresses"),
            pl.col("country").null_count().alias("null_countries"),
        ])
        .collect()
    )
    
    print(f"\n{name}")
    print(result)


get_stats(s1_lazy, "SOURCE 1")
get_stats(s2_lazy, "SOURCE 2")
get_stats(s3_lazy, "SOURCE 3")


SOURCE 1
shape: (1, 4)
┌─────────┬────────────┬────────────────┬────────────────┐
│ rows    ┆ null_names ┆ null_addresses ┆ null_countries │
│ ---     ┆ ---        ┆ ---            ┆ ---            │
│ u32     ┆ u32        ┆ u32            ┆ u32            │
╞═════════╪════════════╪════════════════╪════════════════╡
│ 2206821 ┆ 0          ┆ 0              ┆ 0              │
└─────────┴────────────┴────────────────┴────────────────┘

SOURCE 2
shape: (1, 4)
┌─────────┬────────────┬────────────────┬────────────────┐
│ rows    ┆ null_names ┆ null_addresses ┆ null_countries │
│ ---     ┆ ---        ┆ ---            ┆ ---            │
│ u32     ┆ u32        ┆ u32            ┆ u32            │
╞═════════╪════════════╪════════════════╪════════════════╡
│ 5034616 ┆ 0          ┆ 168967         ┆ 0              │
└─────────┴────────────┴────────────────┴────────────────┘

SOURCE 3
shape: (1, 4)
┌─────────┬────────────┬────────────────┬────────────────┐
│ rows    ┆ null_names ┆ null_addresses ┆ n

In [9]:
def country_stats(lazy_df, name):
    print(f"\n{name}")
    
    result = (
        lazy_df
        .group_by("country")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )
    
    print(result)


country_stats(s1_lazy, "SOURCE 1")
country_stats(s2_lazy, "SOURCE 2")
country_stats(s3_lazy, "SOURCE 3")


SOURCE 1
shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ US      ┆ 1323633 │
│ India   ┆ 883188  │
└─────────┴─────────┘

SOURCE 2
shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ US      ┆ 3016817 │
│ India   ┆ 2017799 │
└─────────┴─────────┘

SOURCE 3
shape: (2, 2)
┌─────────┬─────────┐
│ country ┆ count   │
│ ---     ┆ ---     │
│ str     ┆ u32     │
╞═════════╪═════════╡
│ US      ┆ 3170056 │
│ India   ┆ 2115547 │
└─────────┴─────────┘


In [10]:
gt_lazy = pl.scan_csv(
    GROUND_TRUTH,
    separator="\t",
    infer_schema_length=10000,
)

gt = gt_lazy.collect()

print(gt.shape)
print(gt.head())

(2206821, 2)
shape: (5, 2)
┌───────────────────┬─────────────────────────────────┐
│ source1_entity_id ┆ matched_entity_ids              │
│ ---               ┆ ---                             │
│ str               ┆ str                             │
╞═══════════════════╪═════════════════════════════════╡
│ S1-965667         ┆ S2-681193310,S2-743505751,S3-7… │
│ S1-55344266       ┆ S2-249013014,S2-197070651,S3-4… │
│ S1-343815751      ┆ S2-790675320,S2-479876582,S3-8… │
│ S1-656753428      ┆ S2-153058913,S2-24659151,S3-67… │
│ S1-102811957      ┆ S2-478959098,S2-553508714,S2-6… │
└───────────────────┴─────────────────────────────────┘


In [11]:
import unicodedata
import re

def normalize_unicode(text):
    if text is None:
        return ""
    
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower().strip()
    
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [12]:
def detect_script(text):
    if not text:
        return "empty"
    
    has_devanagari = bool(re.search(r"[\u0900-\u097F]", text))
    has_latin = bool(re.search(r"[a-zA-Z]", text))
    
    if has_devanagari and has_latin:
        return "mixed"
    elif has_devanagari:
        return "devanagari"
    elif has_latin:
        return "latin"
    else:
        return "other"

In [13]:
def detect_script(text):
    if not text:
        return "empty"
    
    has_devanagari = bool(re.search(r"[\u0900-\u097F]", text))
    has_latin = bool(re.search(r"[a-zA-Z]", text))
    
    if has_devanagari and has_latin:
        return "mixed"
    elif has_devanagari:
        return "devanagari"
    elif has_latin:
        return "latin"
    else:
        return "other"

In [14]:
from unidecode import unidecode

def transliterate(text):
    if not text:
        return ""
    
    return unidecode(text).lower().strip()

In [15]:
x = "शिव शक्ति रेस्टोरेंट"

print("Original:     ", x)
print("Transliterated:", transliterate(x))

Original:      शिव शक्ति रेस्टोरेंट
Transliterated: shiv shkti resttorentt


In [16]:
def normalize_columns(df):
    return (
        df
        .with_columns([
            pl.col("business_name")
              .fill_null("")
              .map_elements(normalize_unicode, return_dtype=pl.String)
              .alias("name_norm"),
            
            pl.col("business_address")
              .fill_null("")
              .map_elements(normalize_unicode, return_dtype=pl.String)
              .alias("address_norm"),
        ])
        .with_columns([
            pl.col("name_norm")
              .map_elements(detect_script, return_dtype=pl.String)
              .alias("name_script"),
            
            pl.col("name_norm")
              .map_elements(transliterate, return_dtype=pl.String)
              .alias("name_translit"),
            
            pl.col("address_norm")
              .map_elements(transliterate, return_dtype=pl.String)
              .alias("address_translit"),
        ])
    )